# Router Adapter Inference

This notebook keeps only the thin inference path: bootstrap, optional dependency install, image upload, and `RouterAdapterRuntime` prediction.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')

def refresh_colab_clone(repo_root: Path) -> None:
    repo_root = Path(repo_root)
    if repo_root != CLONE_TARGET or not (repo_root / '.git').exists():
        return
    try:
        subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', REPO_URL], check=False)
        subprocess.run(['git', '-C', str(repo_root), 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', str(repo_root), 'checkout', 'master'], check=True)
        subprocess.run(['git', '-C', str(repo_root), 'pull', '--ff-only', 'origin', 'master'], check=True)
        print(f'Refreshed Colab clone at {repo_root}')
    except Exception as exc:
        print(f'Warning: could not refresh Colab clone at {repo_root}: {exc}')

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'src').is_dir() and (candidate / 'scripts').is_dir():
        ROOT = candidate
        break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    from scripts.colab_repo_bootstrap import install_colab_requirements, mount_drive_if_available, resolve_repo_root, running_in_colab
except ModuleNotFoundError:
    if not (CLONE_TARGET / 'scripts').exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    os.chdir(CLONE_TARGET)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    from scripts.colab_repo_bootstrap import install_colab_requirements, mount_drive_if_available, resolve_repo_root, running_in_colab

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if running_in_colab():
    refresh_colab_clone(ROOT)
    mount_drive_if_available()
    install_colab_requirements(ROOT / 'requirements_colab.txt', in_colab=True)

print(ROOT)

In [ ]:
from scripts.colab_router_adapter_inference import run_inference

IMAGE_PATH = None  # set to a local path if you do not want to use upload

if IMAGE_PATH is None:
    from google.colab import files
    uploaded = files.upload()
    IMAGE_PATH = next(iter(uploaded.keys()))

result = run_inference(IMAGE_PATH, config_env='colab', device='cuda', status_printer=print)
result